# Building Inspector
Введи название здания — получи фото, маску сегментации, вектор признаков и индексы повреждения.

> **Требует:** маски в `data/processed/segmentation-masks-300/{building_name}.png`  
> Как их получить — см. ячейку с комментарием `# Маски`.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# НАСТРОЙКИ
# ═══════════════════════════════════════════════════════════════════════════

IMAGES_DIR   = 'data/processed/inference-402-imgs-v2'
MASKS_DIR    = 'data/processed/masks_inference_402_v2'   # class-index PNG маски
FEATURES_CSV = 'data/interim/400-buildings/building_features_402-32d-b2.csv'
SCORES_CSV   = 'data/interim/400-buildings/facade_damage_scores_lr_4classes-b2.csv'

In [39]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageOps
import cv2

# ── Цветовая схема ────────────────────────────────────────────────────────
CLASSES = [
    'coating_deterioration',
    # 'cracks',
    'masonry_degradation',
    'moisture_bio_damage',
    'vandalism',
]
CLASS_LABELS_RU = [
    'Повреждения\nотделки',
    # 'Трещины',
    'Повреждения\nкладки',
    'Сырость\nбиопоражения',
    'Вандализм',
]
# Цвета совпадают с ноутбуком «3. segformer-inference-analysis.ipynb» (презентация)
CLASS_COLORS = ['#4C9BE8', 
                # '#E8734C',
                '#5CB85C',
                '#9B5DE5',
                '#F4A261']

# Цвета масок — из src/data/visualize_masks.py
MASK_COLORS = {
    0: (30,  30,  30),    # background
    # 1: (217, 21,  54),    # coating_deterioration
    1: (76, 155, 232),
    # 2: (250, 50,  183),   # cracks
    # 2: (209, 125, 42),    # masonry_degradation
    2: (92, 184, 92),
    # 3: (98,  221, 38),    # moisture_bio_damage
    3: (155, 93, 229),
    # 4: (222, 28,  123),   # vandalism
    4: (244, 162, 97)
}
MASK_PATCH_LABELS = [
    'Фон', 'Отделка', 
    # 'Трещины', 
    'Кладка', 'Сырость', 'Вандализм'
]

STATS       = ['mean', 'max', 'prevalence', 'std', 'q75', 'skewness', 'concentration', 'severity_coverage'
               ]
STATS_SHORT = ['μ', 'max', 'prev', 'σ', 'q75', 'skew', 'conc', 'sev\ncov'
               ]

print('Импорты загружены.')

Импорты загружены.


In [40]:
# ── Загрузка таблиц (выполни один раз) ───────────────────────────────────
features_df = pd.read_csv(FEATURES_CSV)
features_df['building_name'] = features_df['building_name'].astype(str)
features_df.set_index('building_name', inplace=True)
features_df.fillna(0.0, inplace=True)
features_df.replace([float('inf'), float('-inf')], 0.0, inplace=True)

scores_df = pd.read_csv(SCORES_CSV)
scores_df['building_name'] = scores_df['building_name'].astype(str)
scores_df.set_index('building_name', inplace=True)

all_buildings = sorted(features_df.index.tolist())

n_feat = len(CLASSES) * len(STATS)
print(f'Зданий: {len(all_buildings)}')
print(f'Вектор признаков: {n_feat}D  ({len(CLASSES)} класса × {len(STATS)} статистик)')

masks_available = os.path.isdir(MASKS_DIR) and len(os.listdir(MASKS_DIR)) > 0
print(f'Маски: {"✓ найдены" if masks_available else "✗ не найдены — панель маски будет пустой"}'
      + (f'  ({len(os.listdir(MASKS_DIR))} файлов)' if masks_available else ''))

Зданий: 402
Вектор признаков: 32D  (4 класса × 8 статистик)
Маски: ✓ найдены  (804 файлов)


In [41]:
# ═══════════════════════════════════════════════════════════════════════════
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ═══════════════════════════════════════════════════════════════════════════

def find_image(building_name):
    for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
        p = os.path.join(IMAGES_DIR, f'{building_name}{ext}')
        if os.path.exists(p):
            return p
    return None


def load_mask(building_name):
    """Загружает class-index PNG маску. Возвращает RGB-массив или None."""
    p = os.path.join(MASKS_DIR, f'{building_name}_mask.png')
    if not os.path.exists(p):
        return None
    index_mask = cv2.imread(p, cv2.IMREAD_GRAYSCALE)   # значения 0–5
    if index_mask is None:
        return None
    color = np.zeros((*index_mask.shape, 3), dtype=np.uint8)
    for cls_id, rgb in MASK_COLORS.items():
        color[index_mask == cls_id] = rgb
    return color


print('Функции определены.')

Функции определены.


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# ГЛАВНАЯ ФУНКЦИЯ ВИЗУАЛИЗАЦИИ
# ═══════════════════════════════════════════════════════════════════════════

def inspect_building(building_name: str):
    name = building_name.strip()
    if not name:
        return

    # ── Проверка наличия ─────────────────────────────────────────────────
    if name not in features_df.index:
        print(f'❌  Здание "{name}" не найдено.')
        print(f'    Примеры: {all_buildings[:5]}')
        return

    feat_row  = features_df.loc[name]
    has_score = name in scores_df.index
    score_row = scores_df.loc[name] if has_score else None

    # ── Загрузка данных ───────────────────────────────────────────────────
    img_path   = find_image(name)
    photo      = ImageOps.exif_transpose(Image.open(img_path).convert('RGB')) if img_path else None
    color_mask = load_mask(name)

    # ── Подготовка данных для графиков ────────────────────────────────────
    # 40D вектор
    feature_values = []
    feature_colors = []
    for cls, color in zip(CLASSES, CLASS_COLORS):
        for stat in STATS:
            feature_values.append(float(feat_row.get(f'{cls}_{stat}', 0.0)))
            feature_colors.append(color)
    feat_x = np.arange(len(feature_values))

    # 5D + GDI
    SCORE_COLS = [f'{c}_score' for c in CLASSES]
    if has_score:
        score_values = [float(score_row.get(c, 0.0)) for c in SCORE_COLS]
        gdi_value    = float(score_row.get('Global_Damage_Index', 0.0))
    else:
        score_values = [0.0] * len(CLASSES)
        gdi_value    = 0.0

    score_x      = np.arange(len(CLASSES) + 1)
    score_colors = CLASS_COLORS + ['#D4A017']
    score_vals   = score_values + [gdi_value]
    score_labels = CLASS_LABELS_RU + ['Глобальный\nиндекс']

    # ── Фигура ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 13), facecolor='white')
    gs  = gridspec.GridSpec(
        2, 2, figure=fig,
        height_ratios=[2.2, 1],
        hspace=0.01, wspace=0.01
    )

    # ── [0, 0]  Оригинальное фото ─────────────────────────────────────────
    ax_photo = fig.add_subplot(gs[0, 0])
    if photo is not None:
        ax_photo.imshow(photo)
    else:
        ax_photo.set_facecolor('#E0E0E0')
        ax_photo.text(0.5, 0.5, f'Фото не найдено\n{name}',
                      ha='center', va='center', transform=ax_photo.transAxes,
                      fontsize=13, color='#666666')
    ax_photo.set_title('Исходное фото', fontsize=14, fontweight='bold',
                        pad=10, color='#222222')
    ax_photo.axis('off')
    num_tiles = int(feat_row.get('num_tiles', 0))
    ax_photo.text(
        0.015, 0.975, f'{name}   (тайлов: {num_tiles})',
        transform=ax_photo.transAxes, ha='left', va='top', fontsize=9.5,
        color='white',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='#111111', alpha=0.72, edgecolor='none')
    )

    # ── [0, 1]  Маска сегментации ─────────────────────────────────────────
    ax_mask = fig.add_subplot(gs[0, 1])
    if color_mask is not None:
        ax_mask.imshow(color_mask)
        patches = [
            mpatches.Patch(
                color=tuple(v / 255 for v in MASK_COLORS[i]),
                label=MASK_PATCH_LABELS[i]
            )
            for i in range(5)
        ]
        ax_mask.legend(
            handles=patches, loc='lower right', fontsize=9,
            frameon=True, facecolor='white', framealpha=0.85, edgecolor='none'
        )
    else:
        ax_mask.set_facecolor('#1A1A2E')
        ax_mask.text(
            0.5, 0.5,
            'Маска не найдена.\n'
            'Сохрани маски в Kaggle и положи\n'
            'в data/processed/segmentation-masks-300/',
            ha='center', va='center', transform=ax_mask.transAxes,
            fontsize=11, color='#AAAAAA', linespacing=1.7
        )
    ax_mask.set_title('Маска сегментации (SegFormer)', fontsize=14,
                       fontweight='bold', pad=10, color='#222222')
    ax_mask.axis('off')

    # ── [1, 0]  40D вектор признаков ──────────────────────────────────────
    ax_feat = fig.add_subplot(gs[1, 0])
    ax_feat.set_facecolor('#FAFAFA')

    ax_feat.bar(
        feat_x, feature_values,
        color=feature_colors, width=0.75,
        edgecolor='white', linewidth=0.6, zorder=3
    )

    # Разделители между группами классов
    for i in range(1, len(CLASSES)):
        ax_feat.axvline(
            x=i * len(STATS) - 0.5,
            color='#CCCCCC', linewidth=1.2, linestyle='--', zorder=2
        )

    # Подписи групп (снизу, под тиками)
    for i, (lbl, color) in enumerate(zip(CLASS_LABELS_RU, CLASS_COLORS)):
        mid = i * len(STATS) + (len(STATS) - 1) / 2
        ax_feat.annotate(
            lbl, xy=(mid, 0), xytext=(mid, -0.21),
            xycoords=('data', 'axes fraction'),
            textcoords=('data', 'axes fraction'),
            ha='center', va='top', fontsize=7.5, fontweight='bold',
            color=color, annotation_clip=False
        )

    ax_feat.set_xticks(feat_x)
    ax_feat.set_xticklabels(STATS_SHORT * len(CLASSES), fontsize=7)
    ax_feat.set_ylabel('Значение', fontsize=10, color='#555555')
    ax_feat.set_title(
        f'Вектор сырых статистик  ({len(feature_values)}D '
        f'= {len(CLASSES)} класса × {len(STATS)} признаков)',
        fontsize=12, fontweight='bold', pad=8, color='#222222'
    )
    ax_feat.tick_params(axis='y', labelsize=8)
    ax_feat.grid(axis='y', linestyle='--', alpha=0.45, zorder=0)
    ax_feat.spines[['top', 'right']].set_visible(False)
    ax_feat.set_xlim(-0.6, len(feature_values) - 0.4)

    # ── [1, 1]  Индексы повреждения ───────────────────────────────────────
    ax_score = fig.add_subplot(gs[1, 1])
    ax_score.set_facecolor('#FAFAFA')

    bars_s = ax_score.bar(
        score_x, score_vals,
        color=score_colors, width=0.6,
        edgecolor='white', linewidth=1.0, zorder=3
    )

    # Разделитель перед GDI
    ax_score.axvline(x=len(CLASSES) - 0.5, color='#999999',
                     linewidth=1.2, linestyle=':', zorder=2)

    # Значения над барами
    y_pad = (max(score_vals) if max(score_vals) > 0 else 0.1) * 0.025
    for bar, val in zip(bars_s, score_vals):
        ax_score.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + y_pad,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='#333333'
        )

    ax_score.set_xticks(score_x)
    ax_score.set_xticklabels(score_labels, fontsize=9.5)
    ax_score.set_ylim(0, max(min(max(score_vals) * 1.25 + 0.05, 1.18), 0.15))
    ax_score.set_ylabel('Damage Score  [0 – 1]', fontsize=10, color='#555555')
    ax_score.set_title(
        'Калиброванные индексы повреждения  (LogReg)',
        fontsize=12, fontweight='bold', pad=6, color='#222222'
    )
    ax_score.tick_params(axis='y', labelsize=8)
    ax_score.grid(axis='y', linestyle='--', alpha=0.45, zorder=0)
    ax_score.spines[['top', 'right']].set_visible(False)

    if not has_score:
        ax_score.text(0.5, 0.5, 'Индексы отсутствуют в CSV',
                      ha='center', va='center', transform=ax_score.transAxes,
                      fontsize=11, color='#888888')

    plt.suptitle(
        f'Инспектор здания:  {name}',
        fontsize=16, fontweight='bold', y=0.995, color='#111111'
    )
    plt.show()


print('inspect_building() определена.')

inspect_building() определена.


In [42]:
# ═══════════════════════════════════════════════════════════════════════════
# ГЛАВНАЯ ФУНКЦИЯ ВИЗУАЛИЗАЦИИ
# ═══════════════════════════════════════════════════════════════════════════

def inspect_building(building_name: str):
    name = building_name.strip()
    if not name:
        return

    # ── Проверка наличия ─────────────────────────────────────────────────
    if name not in features_df.index:
        print(f'❌  Здание "{name}" не найдено.')
        print(f'    Примеры: {all_buildings[:5]}')
        return

    feat_row  = features_df.loc[name]
    has_score = name in scores_df.index
    score_row = scores_df.loc[name] if has_score else None

    # ── Загрузка данных ───────────────────────────────────────────────────
    img_path   = find_image(name)
    photo      = ImageOps.exif_transpose(Image.open(img_path).convert('RGB')) if img_path else None
    color_mask = load_mask(name)

    # ── Подготовка данных для графиков ────────────────────────────────────
    # 4D
    SCORE_COLS = [f'{c}_score' for c in CLASSES]
    if has_score:
        score_values = [float(score_row.get(c, 0.0)) for c in SCORE_COLS]      
    else:
        score_values = [0.0] * len(CLASSES)      

    score_x      = np.arange(len(CLASSES))
    score_colors = CLASS_COLORS
    score_vals   = score_values
    score_labels = CLASS_LABELS_RU

    # ── Фигура ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 7), facecolor='white')

    gs = gridspec.GridSpec(
        1, 3, figure=fig,
        wspace=0.2
    )

    # ── [0, 0]  Оригинальное фото ─────────────────────────────────────────
    ax_photo = fig.add_subplot(gs[0, 0])
    if photo is not None:
        ax_photo.imshow(photo)
    else:
        ax_photo.set_facecolor('#E0E0E0')
        ax_photo.text(0.5, 0.5, f'Фото не найдено\n{name}',
                      ha='center', va='center', transform=ax_photo.transAxes,
                      fontsize=13, color='#666666')
    ax_photo.set_title('Исходное фото', fontsize=14, fontweight='bold',
                        pad=10, color='#222222')
    ax_photo.axis('off')
    num_tiles = int(feat_row.get('num_tiles', 0))
    ax_photo.text(
        0.015, 0.975, f'{name}   (тайлов: {num_tiles})',
        transform=ax_photo.transAxes, ha='left', va='top', fontsize=9.5,
        color='white',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='#111111', alpha=0.72, edgecolor='none')
    )

    # ── [0, 1]  Маска сегментации ─────────────────────────────────────────
    ax_mask = fig.add_subplot(gs[0, 1])
    if color_mask is not None:
        ax_mask.imshow(color_mask)
        patches = [
            mpatches.Patch(
                color=tuple(v / 255 for v in MASK_COLORS[i]),
                label=MASK_PATCH_LABELS[i]
            )
            for i in range(5)
        ]
        ax_mask.legend(
            handles=patches, loc='lower right', fontsize=9,
            frameon=True, facecolor='white', framealpha=0.85, edgecolor='none'
        )
    else:
        ax_mask.set_facecolor('#1A1A2E')
        ax_mask.text(
            0.5, 0.5,
            'Маска не найдена.\n'
            'Сохрани маски в Kaggle и положи\n'
            'в data/processed/segmentation-masks-300/',
            ha='center', va='center', transform=ax_mask.transAxes,
            fontsize=11, color='#AAAAAA', linespacing=1.7
        )
    ax_mask.set_title('Маска сегментации', fontsize=14,
                       fontweight='bold', pad=10, color='#222222')
    ax_mask.axis('off')

        # ── [1, 1]  Индексы повреждения ───────────────────────────────────────
    ax_score = fig.add_subplot(gs[0, 2])
    ax_score.set_box_aspect(0.7)
    ax_score.set_facecolor('#FAFAFA')

    bars_s = ax_score.bar(
        score_x, score_vals,
        color=score_colors, width=0.6,
        edgecolor='white', linewidth=1.0, zorder=3
    )  

    # Значения над барами
    y_pad = (max(score_vals) if max(score_vals) > 0 else 0.1) * 0.025
    for bar, val in zip(bars_s, score_vals):
        ax_score.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + y_pad,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='#333333'
        )

    ax_score.set_xticks(score_x)
    ax_score.set_xticklabels(score_labels, fontsize=9.5)
    ax_score.set_ylim(0, max(min(max(score_vals) * 1.25 + 0.05, 1.18), 0.15))
    ax_score.set_ylabel('Damage Score  [0 – 1]', fontsize=10, color='#555555')
    ax_score.set_title(
        'Индексы повреждения',
        fontsize=12, fontweight='bold', pad=6, color='#222222'
    )
    ax_score.tick_params(axis='y', labelsize=8)
    ax_score.grid(axis='y', linestyle='--', alpha=0.45, zorder=0)
    ax_score.spines[['top', 'right']].set_visible(False)

    if not has_score:
        ax_score.text(0.5, 0.5, 'Индексы отсутствуют в CSV',
                      ha='center', va='center', transform=ax_score.transAxes,
                      fontsize=11, color='#888888')

    plt.suptitle(
        f'Инспектор здания:  {name}',
        fontsize=16, fontweight='bold', y=0.995, color='#111111'
    )
    plt.show()


print('inspect_building() определена.')

inspect_building() определена.


In [43]:
# ═══════════════════════════════════════════════════════════════════════════
# ИНТЕРАКТИВНЫЙ ВИДЖЕТ
# ═══════════════════════════════════════════════════════════════════════════

txt = widgets.Text(
    placeholder='Введи название здания, например: DSC02518',
    description='Здание:',
    layout=widgets.Layout(width='520px'),
    style={'description_width': '70px'}
)
btn = widgets.Button(
    description='Показать',
    button_style='primary',
    icon='search'
)
out = widgets.Output()

def _run(_=None):
    with out:
        clear_output(wait=True)
        inspect_building(txt.value)

btn.on_click(_run)
txt.on_submit(_run)    # Enter тоже работает

display(widgets.VBox([widgets.HBox([txt, btn]), out]))

/tmp/ipykernel_53980/773917257.py:24: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  txt.on_submit(_run)    # Enter тоже работает
